# Fresh SAM Mask Generation & Visual Rendering Pipeline
This notebook runs Meta's SAM (ViT-B) with CUDA GPU acceleration from scratch across the PCB dataset:
1. Loads SAM ViT-B on GPU.
2. Feeds component bounding boxes to SAM to generate fresh high-fidelity masks.
3. Extracts polygon boundary points and creates standard LabelMe `.json` files.
4. Renders filled semi-transparent color-coded mask overlays onto the original images using OpenCV.
5. Packages all JSONs and visual images into a single zip file for direct local download.

In [ ]:
# Install Segment Anything
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import glob
import json
import time
import shutil
import urllib.request
from pathlib import Path
import cv2
import numpy as np
import torch
from segment_anything import sam_model_registry, SamPredictor

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Download SAM checkpoint if not present
sam_ckpt = "/kaggle/working/sam_vit_b.pth"
found_sam = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if 'sam_vit_b' in f and f.endswith('.pth'):
            found_sam = os.path.join(root, f)
            break

if found_sam:
    sam_ckpt = found_sam
    print(f"Using existing SAM checkpoint: {sam_ckpt}")
elif not os.path.exists(sam_ckpt):
    print("Downloading SAM ViT-B checkpoint (~375 MB)...\n")
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", sam_ckpt)
    print("Downloaded successfully!")

In [ ]:
# Class mapping and colors (BGR)
CLASS_MAP = {
    0: 'Cap1', 1: 'Cap2', 2: 'Cap3', 3: 'Cap4',
    4: 'MOSFET', 5: 'Mov', 6: 'Resistor', 7: 'Transformer'
}
CLASS_COLORS = {
    "Cap1":        (30, 180, 255),   # orange
    "Cap2":        (255, 180, 30),   # blue
    "Cap3":        (50, 220, 50),    # green
    "Cap4":        (180, 50, 255),   # magenta
    "MOSFET":      (0, 0, 220),      # red
    "Mov":         (220, 220, 0),    # cyan
    "Resistor":    (0, 220, 220),    # yellow
    "Transformer": (200, 100, 255),  # pink
}
DEFAULT_COLOR = (200, 200, 200)

# Prepare output folders
WORK_DIR = Path("/kaggle/working/Fresh_Output")
JSON_DIR = WORK_DIR / "2_SAM_JSON_Output"
VIS_DIR  = WORK_DIR / "3_Script_Visual_Output"
JSON_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

# Load SAM Predictor on CUDA GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Initializing SAM ViT-B on {device}...")
sam = sam_model_registry["vit_b"](checkpoint=sam_ckpt)
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)
print("SAM ViT-B loaded and ready!")

In [ ]:
# Locate all images and labels in dataset inputs
print("Locating input datasets...")
all_txt_files = list(Path("/kaggle/input").rglob("*.txt"))
label_files = [p for p in all_txt_files if p.parent.name == 'labels' and 'yaml' not in p.name.lower()]
print(f"Found {len(label_files)} label files with annotations.")

def parse_yolo_boxes(txt_path, w, h):
    boxes = []
    with open(txt_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cid = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = max(0, int((xc - bw/2) * w))
            y1 = max(0, int((yc - bh/2) * h))
            x2 = min(w, int((xc + bw/2) * w))
            y2 = min(h, int((yc + bh/2) * h))
            if x2 > x1 + 4 and y2 > y1 + 4:
                boxes.append((cid, [x1, y1, x2, y2]))
    return boxes

processed_count = 0
t0 = time.time()

for idx, lbl_p in enumerate(label_files, 1):
    stem = lbl_p.stem
    parent_split = lbl_p.parent.parent
    img_candidates = list((parent_split / 'images').glob(f"{stem}.*"))
    if not img_candidates:
        img_candidates = list(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    if not img_candidates:
        continue
        
    img_p = img_candidates[0]
    img = cv2.imread(str(img_p))
    if img is None: continue
    h, w = img.shape[:2]
    
    boxes = parse_yolo_boxes(lbl_p, w, h)
    if not boxes: continue
    
    # Run SAM on GPU
    predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    overlay = img.copy()
    shapes = []
    
    for cid, box in boxes:
        cls_name = CLASS_MAP.get(cid, f"Class_{cid}")
        color = CLASS_COLORS.get(cls_name, DEFAULT_COLOR)
        
        masks, scores, _ = predictor.predict(box=np.array(box)[None, :], multimask_output=False)
        m_uint = masks[0].astype(np.uint8)
        contours, _ = cv2.findContours(m_uint, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if contours:
            cnt = max(contours, key=cv2.contourArea)
            epsilon = 0.005 * cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, epsilon, True)
            pts = [[round(float(p[0][0]), 2), round(float(p[0][1]), 2)] for p in approx]
            pts_np = np.array(pts, np.int32)
        else:
            # Fallback to box polygon
            x1, y1, x2, y2 = box
            pts = [[float(x1), float(y1)], [float(x2), float(y1)], [float(x2), float(y2)], [float(x1), float(y2)]]
            pts_np = np.array(pts, np.int32)
            
        cv2.fillPoly(overlay, [pts_np], color)
        cv2.polylines(overlay, [pts_np], isClosed=True, color=color, thickness=2)
        
        shapes.append({
            "label": cls_name,
            "points": pts,
            "group_id": None,
            "description": "",
            "shape_type": "polygon",
            "flags": {}
        })
        
    # Blend semi-transparent masks
    blended = cv2.addWeighted(overlay, 0.45, img, 0.55, 0)
    cv2.imwrite(str(VIS_DIR / f"{stem}-masked.jpg"), blended, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    
    # Save LabelMe JSON
    labelme_data = {
        "version": "5.2.1",
        "flags": {},
        "shapes": shapes,
        "imagePath": img_p.name,
        "imageData": None,
        "imageHeight": h,
        "imageWidth": w
    }
    with open(JSON_DIR / f"{stem}.json", "w", encoding="utf-8") as jf:
        json.dump(labelme_data, jf, indent=2)
        
    processed_count += 1
    if idx % 100 == 0 or idx == len(label_files):
        elapsed = time.time() - t0
        print(f"  Processed [{idx}/{len(label_files)}] in {elapsed:.1f}s ({processed_count} completed)")

print(f"\nAll fresh SAM masks generated! Total: {processed_count}")

In [ ]:
# Zip the entire output package for easy one-click download
zip_target = "/kaggle/working/Fresh_Datasets_SAM_and_Visuals"
print("Zipping fresh generated datasets...")
shutil.make_archive(zip_target, 'zip', str(WORK_DIR))
print(f"Created: {zip_target}.zip")
print(f"Zip size: {os.path.getsize(zip_target + '.zip') / (1024*1024):.2f} MB")
print("Done! Ready for download.")